Graph Setup

In [24]:
import networkx as nx

G = nx.read_edgelist("data/CA-AstroPh.txt", comments="#", nodetype=int)

# --- Controllo e rimozione dei self-loop ---
n_self_loops = nx.number_of_selfloops(G)
print(f"Self-loop trovati: {n_self_loops}")

if n_self_loops > 0:
    G.remove_edges_from(nx.selfloop_edges(G))
    print(f"Self-loop rimossi. Nuovo numero di archi: {G.number_of_edges()}")

# G ripulito.
# --- Componente connessa più grande ---

largest_cc = max(nx.connected_components(G), key=len)
G_lcc = G.subgraph(largest_cc).copy()

Self-loop trovati: 60
Self-loop rimossi. Nuovo numero di archi: 198050


In [25]:
from algorithms.cost_functions import cost_random, cost_degree
from algorithms.cs_greedy import cost_seeds_greedy
from algorithms.wtss import wtss
from algorithms.my_seeds import my_seeds
from algorithms.cascade import simulate_cascade

# Le due funzioni di costo, calcolate una sola volta su G_lcc
c1 = cost_random(G_lcc, low=1, high=21, seed=42)
c2 = cost_degree(G_lcc)

print(f"c1 - costo totale: {sum(c1.values())}, media: {sum(c1.values())/len(c1):.2f}")
print(f"c2 - costo totale: {sum(c2.values())}, media: {sum(c2.values())/len(c2):.2f}")

c1 - costo totale: 197720, media: 11.04
c2 - costo totale: 201526, media: 11.26


In [26]:
algorithms = {
    "Greedy-f1": lambda G, k, c: cost_seeds_greedy(G, k, c, "f1"),
    "Greedy-f2": lambda G, k, c: cost_seeds_greedy(G, k, c, "f2"),
    "Greedy-f3": lambda G, k, c: cost_seeds_greedy(G, k, c, "f3"),
    "WTSS":      lambda G, k, c: wtss(G, k, c),
    "My-Seeds":  lambda G, k, c: my_seeds(G, k, c),
}

percentages = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20]

cost_scenarios = {
    "c1_random": c1,
    "c2_degree": c2,
}

k_values = {}
for cost_name, c in cost_scenarios.items():
    total_cost = sum(c.values())
    k_values[cost_name] = [round(p * total_cost) for p in percentages]
    print(f"{cost_name} (totale={total_cost}): k = {k_values[cost_name]}")

c1_random (totale=197720): k = [989, 1977, 3954, 9886, 19772, 39544]
c2_degree (totale=201526): k = [1008, 2015, 4031, 10076, 20153, 40305]


Esperimento 1

In [27]:
import time
start = time.time()
S = cost_seeds_greedy(G_lcc, k_values["c2_degree"][-1], c2, "f1")
print(f"Tempo: {time.time()-start:.2f}s, |S| = {len(S)}")

Tempo: 267.26s, |S| = 3510
